# Tent reproduction (Student A) on Colab

Re-runs the three upstream configs (`source`, `norm`, `tent`) on both WRN architectures and two seeds, then builds the README-format tables, the severity-trend plot, the seed-variance summary, and a deviation report.

**Environment.** The upstream pins are a Python-3.8 / 2020-era set (`torch==1.8.1`; robustbench v0.1 drags in `numpy~=1.19.4`), which will not build on Colab's stock Python 3.11+. We make a throwaway Python-3.8 env with `uv` and install `requirements.txt` *unchanged* inside it, so the reproduction stays faithful. The one unavoidable deviation is `torch 1.8.1+cu111` (vs the authors' cu102) — `deviation_report.md` flags any resulting wobble.

**Downloads.** RobustBench v0.1 fetches its checkpoints *and* the CIFAR-10-C arrays from Google Drive with a downloader that predates Google's confirmation page, so it silently saves HTML instead of the real files. This notebook routes around that: **modern `gdown` for the two checkpoints** and the **official Zenodo tarball for CIFAR-10-C**. Everything is pre-fetched before the runs so they don't stall mid-matrix.

**Paths** (edit `DRIVE_ROOT` / repo coordinates in cell 1 if yours differ):

| what | path |
|---|---|
| repo clone | `/content/FRMDL-Tent-Reproducibility` (branch `reproducibility`) |
| Python 3.8 venv | `/content/venv` |
| CIFAR-10-C + checkpoints | **Drive** `…/frmdl_tent/{data,ckpt}` (symlinked into `tent/`, fetched once) |
| logs + results | **Drive** `…/frmdl_tent/output_A` |

> Run cells **top to bottom.** The shell cells interpolate Python variables with `{...}`, so cell 1 must run first. Full matrix ≈ 2–4 h on a T4; logs stream to Drive and the run is resumable (`SKIP_EXISTING=1`).

## 0. Confirm the GPU runtime
Runtime ▸ Change runtime type ▸ **T4 GPU**, then:

In [ ]:
!nvidia-smi -L

## 1. Mount Drive and define paths
Every later cell reads these Python variables.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

# --- edit these if your repo / Drive layout differs -----------------------
GITHUB_USER = 'athxrva02'
GITHUB_REPO = 'FRMDL-Tent-Reproducibility'
BRANCH      = 'reproducibility'
DRIVE_ROOT  = '/content/drive/MyDrive/frmdl_tent'
# --------------------------------------------------------------------------

REPO_DIR = f'/content/{GITHUB_REPO}'
TENT_DIR = f'{REPO_DIR}/tent'
VENV     = '/content/venv'
OUT_ROOT = f'{DRIVE_ROOT}/output_A'
DATA_DIR = f'{DRIVE_ROOT}/data'
CKPT_DIR = f'{DRIVE_ROOT}/ckpt'
REPO_URL = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git'

for d in (OUT_ROOT, DATA_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
print('paths ready ->', OUT_ROOT)

## 2. Clone (or update) the repo
Uses `subprocess` with an argument **list** — no shell, so the URL and destination can't get glued together. `repro_a/` must already be committed and pushed to `reproducibility`. For a **private** repo, set `REPO_URL = f'https://{GITHUB_USER}:<PAT>@github.com/{GITHUB_USER}/{GITHUB_REPO}.git'`.

In [ ]:
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

print(sorted(os.listdir(f'{TENT_DIR}/repro_a')))

## 3. Persist data + checkpoints on Drive
Symlink `tent/data` and `tent/ckpt` to Drive so CIFAR-10-C and the checkpoints live there and are fetched only once. `cifar10c.py` uses the default `./data` / `./ckpt`, which now resolve to Drive.

In [ ]:
subprocess.run(['ln', '-sfn', DATA_DIR, f'{TENT_DIR}/data'], check=True)
subprocess.run(['ln', '-sfn', CKPT_DIR, f'{TENT_DIR}/ckpt'], check=True)
!ls -ld {TENT_DIR}/data {TENT_DIR}/ckpt

## 4. Fetch both checkpoints with modern `gdown`
RobustBench's own Drive downloader saves HTML confirmation pages instead of the weights (`UnpicklingError: invalid load key, '<'`). `gdown` handles the confirmation flow. RobustBench skips its download when the `.pt` already exists, so these pre-fetched files are used as-is. (Re-downloads anything smaller than 1 MB, which catches a stale HTML page.)

In [ ]:
!pip install -q -U gdown

CKPT_CORR = f'{CKPT_DIR}/cifar10/corruptions'
os.makedirs(CKPT_CORR, exist_ok=True)

CHECKPOINTS = {
    'Standard':                '1t98aEuzeTL8P7Kpd5DIrCoCL21BNZUhC',
    'Hendrycks2020AugMix_WRN': '1wy7gSRsUZzCzj8QhmTbcnwmES_2kkNph',
}
for name, gid in CHECKPOINTS.items():
    dst = f'{CKPT_CORR}/{name}.pt'
    if (not os.path.exists(dst)) or os.path.getsize(dst) < 1_000_000:
        subprocess.run(['gdown', gid, '-O', dst], check=True)
    print(f'{name}: {os.path.getsize(dst) // (1024*1024)} MB')

# sanity: load each with the venv's torch (built after cell 6 -- safe to re-run then)
# !{VENV}/bin/python -c "import torch,glob; [torch.load(p, map_location='cpu') for p in glob.glob('{CKPT_CORR}/*.pt')]; print('checkpoints OK')"

## 5. Fetch CIFAR-10-C from Zenodo
The CIFAR-10-C `.npy` files also come from Drive in robustbench v0.1 and hit the same corruption (`Cannot load file containing pickled data`). We pull the official Zenodo tarball (~2.9 GB) instead and drop the arrays where robustbench expects them (`<DATA_DIR>/cifar10c/*.npy`). Idempotent — skips if already there.

In [ ]:
import numpy as np

CIFAR_C = f'{DATA_DIR}/cifar10c'

def _loadable(path):
    try:
        np.load(path, mmap_mode='r'); return True
    except Exception:
        return False

# Only re-download if the arrays are missing OR corrupt (a stale HTML page from
# robustbench's Drive downloader still 'exists' but won't np.load).
ok = _loadable(f'{CIFAR_C}/labels.npy') and _loadable(f'{CIFAR_C}/gaussian_noise.npy')
if not ok:
    subprocess.run(['rm', '-rf', CIFAR_C], check=False)
    os.makedirs(CIFAR_C, exist_ok=True)
    tar = '/content/CIFAR-10-C.tar'
    subprocess.run(['rm', '-f', tar], check=False)
    subprocess.run(['wget', '-O', tar,
        'https://zenodo.org/records/2535967/files/CIFAR-10-C.tar?download=1'], check=True)
    gb = os.path.getsize(tar) / 1e9
    assert gb > 2.0, f'download is only {gb:.2f} GB -> got an HTML/error page, not the tar'
    # top folder in the tarball is 'CIFAR-10-C/'; strip it so files land in cifar10c/
    subprocess.run(['tar', '-xf', tar, '-C', CIFAR_C, '--strip-components=1'], check=True)

# prove the arrays load (exactly what robustbench does at run time)
assert _loadable(f'{CIFAR_C}/labels.npy') and _loadable(f'{CIFAR_C}/gaussian_noise.npy'), \
    'CIFAR-10-C .npy still not loadable -- inspect the download above'
print('CIFAR-10-C OK:',
      np.load(f'{CIFAR_C}/gaussian_noise.npy', mmap_mode='r').shape,
      '|', len([f for f in os.listdir(CIFAR_C) if f.endswith('.npy')]), 'npy files')

## 6. Build the Python 3.8 environment
`uv` fetches Python 3.8 and installs the upstream pins into `/content/venv`. `torch 1.8.1+cu111` satisfies the `==1.8.1` pin and runs on the T4. (~3–5 min.)

In [ ]:
!pip install -q uv
!uv venv --python 3.8 {VENV}
!uv pip install --python {VENV}/bin/python \
    torch==1.8.1+cu111 torchvision==0.9.1+cu111 \
    -f https://download.pytorch.org/whl/torch_stable.html
!uv pip install --python {VENV}/bin/python -r {TENT_DIR}/requirements.txt

Verify the 3.8 env sees the GPU and can load the checkpoints:

In [ ]:
!{VENV}/bin/python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"
!{VENV}/bin/python -c "import torch, glob; [torch.load(p, map_location='cpu') for p in glob.glob('{CKPT_CORR}/*.pt')]; print('checkpoints load OK')"

## 7. Smoke test (~2 min)
One tiny eval to confirm the whole path works end-to-end. `run_all.sh` `cd`s into `tent/` itself, so an absolute path is fine.

In [ ]:
!PY={VENV}/bin/python OUT_ROOT={OUT_ROOT} bash {TENT_DIR}/repro_a/run_all.sh --smoke

## 8. Run the full 12-run matrix
2 archs × 3 methods × 2 seeds. Logs stream to Drive; `SKIP_EXISTING=1` makes it resumable — if Colab disconnects, just re-run this cell and finished runs are skipped.

> **If a run ever crashes mid-way**, its dir holds a partial log that `SKIP_EXISTING=1` would wrongly skip. Clear partials before resuming — the commented line does that for the smoke + any incomplete dirs (a complete run's log ends with 15 `error %` lines).

In [ ]:
# !rm -rf {OUT_ROOT}/_smoke   # uncomment to clear the smoke-test dir before the real run
!PY={VENV}/bin/python OUT_ROOT={OUT_ROOT} SKIP_EXISTING=1 bash {TENT_DIR}/repro_a/run_all.sh

### (Optional) Fast path — headline WRN-28-10 numbers first
Runs `source`/`norm`/`tent` on `Standard`, seed 1, to get the 18.6 figure before committing hours to the full matrix. The analysis cells work on this partial tree.

In [ ]:
os.chdir(TENT_DIR)
for m in ['source', 'norm', 'tent']:
    !{VENV}/bin/python cifar10c.py --cfg cfgs/{m}.yaml RNG_SEED 1 SAVE_DIR {OUT_ROOT}/Standard/{m}/seed1

## 9. Build the deliverables
`parse_logs.py` is stdlib-only and `make_tables.py` needs only pandas + matplotlib (preinstalled in Colab's base Python), so analysis runs outside the 3.8 venv.

In [ ]:
!python {TENT_DIR}/repro_a/parse_logs.py  --root {OUT_ROOT} --out {OUT_ROOT}/results.csv
!python {TENT_DIR}/repro_a/make_tables.py --csv  {OUT_ROOT}/results.csv --out {OUT_ROOT}

## 10. View the results
**Reproduction sanity:** WRN-28-10 `tent` severity-5 mean should be **18.6 ± ~1pp**; >2pp is flagged automatically (likely cause: cu111 vs the authors' cuDNN/torch build).

In [ ]:
from IPython.display import Markdown, Image, display
for name in ('table_sev5_Standard.md', 'table_sev5_Hendrycks2020AugMix_WRN.md',
             'severity_trend.md', 'variance.md', 'deviation_report.md'):
    p = f'{OUT_ROOT}/{name}'
    if os.path.exists(p):
        display(Markdown(open(p).read()))
trend = f'{OUT_ROOT}/severity_trend.png'
if os.path.exists(trend):
    display(Image(trend))